# GPT-5 Chat Fine-Tuning Notebook

This notebook does the following:

1. Loads a chat-format JSONL dataset
2. Validates each example
3. Splits it into:
   - 2000 training examples
   - 500 validation examples
   - 200 test examples
4. Uploads the training and validation files for fine-tuning
5. Creates a supervised fine-tuning job on `gpt-5-chat`
6. Monitors the job
7. Evaluates the final fine-tuned model on the held-out 200-example test set

> **Expected input format**: each JSONL line should look like:
>
> ```json
> {"messages":[
>   {"role":"system","content":"..."},
>   {"role":"user","content":"..."},
>   {"role":"assistant","content":"..."}
> ]}
> ```

In [ ]:
# Optional: install/upgrade the OpenAI SDK in Jupyter if needed
# Uncomment if your environment does not already have the latest SDK.
# %pip install -U openai

In [ ]:
import json
import random
import time
from pathlib import Path
from collections import Counter

try:
    from openai import OpenAI
except ImportError:
    raise ImportError(
        "OpenAI Python SDK not found. Run: %pip install -U openai"
    )

## 1) Configuration

Update `INPUT_JSONL` to point at your full dataset.

In [ ]:
# ----------------------------
# CONFIG
# ----------------------------
SEED = 42

TRAIN_SIZE = 2000
VALID_SIZE = 500
TEST_SIZE = 200

# Change this to your source dataset path
INPUT_JSONL = "all_examples.jsonl"

OUTPUT_DIR = Path("ft_split_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = OUTPUT_DIR / "train.jsonl"
VALID_PATH = OUTPUT_DIR / "valid.jsonl"
TEST_PATH  = OUTPUT_DIR / "test.jsonl"

PROJECT_NAME = "module6"
BASE_MODEL = "gpt-5-chat"

## 2) Helpers

In [ ]:
def load_jsonl(path: str):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON on line {line_no}: {e}")
    return rows


def save_jsonl(rows, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def validate_chat_example(example, idx):
    if not isinstance(example, dict):
        raise ValueError(f"Example {idx} is not a dict.")

    if "messages" not in example:
        raise ValueError(f"Example {idx} missing 'messages'.")

    messages = example["messages"]
    if not isinstance(messages, list) or len(messages) == 0:
        raise ValueError(f"Example {idx} has invalid 'messages'.")

    valid_roles = {"system", "user", "assistant", "developer"}

    for m_i, msg in enumerate(messages):
        if not isinstance(msg, dict):
            raise ValueError(f"Example {idx}, message {m_i} is not a dict.")
        if "role" not in msg or "content" not in msg:
            raise ValueError(f"Example {idx}, message {m_i} missing role/content.")
        if msg["role"] not in valid_roles:
            raise ValueError(
                f"Example {idx}, message {m_i} has invalid role: {msg['role']}"
            )
        if not isinstance(msg["content"], str):
            raise ValueError(
                f"Example {idx}, message {m_i} content must be a string."
            )


def preview_jsonl(path, n=2, max_chars=1500):
    print(f"\nPreview: {path}")
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            obj = json.loads(line)
            text = json.dumps(obj, indent=2, ensure_ascii=False)
            print(text[:max_chars])
            print("-" * 80)


def split_prompt_and_target(example):
    messages = example["messages"]
    if messages[-1]["role"] != "assistant":
        raise ValueError("Last message must be assistant for evaluation.")
    prompt_messages = messages[:-1]
    target = messages[-1]["content"]
    return prompt_messages, target

## 3) Load, validate, shuffle, and split

In [ ]:
examples = load_jsonl(INPUT_JSONL)

for i, ex in enumerate(examples):
    validate_chat_example(ex, i)

required_total = TRAIN_SIZE + VALID_SIZE + TEST_SIZE
actual_total = len(examples)

print("Total examples found:", actual_total)

if actual_total < required_total:
    raise RuntimeError(
        f"You need at least {required_total} examples before splitting. "
        f"Current count: {actual_total}"
    )

random.seed(SEED)
random.shuffle(examples)

train_data = examples[:TRAIN_SIZE]
valid_data = examples[TRAIN_SIZE:TRAIN_SIZE + VALID_SIZE]
test_data  = examples[TRAIN_SIZE + VALID_SIZE:TRAIN_SIZE + VALID_SIZE + TEST_SIZE]

save_jsonl(train_data, TRAIN_PATH)
save_jsonl(valid_data, VALID_PATH)
save_jsonl(test_data, TEST_PATH)

print(f"Saved train: {TRAIN_PATH} ({len(train_data)} examples)")
print(f"Saved valid: {VALID_PATH} ({len(valid_data)} examples)")
print(f"Saved test : {TEST_PATH} ({len(test_data)} examples)")

## 4) Quick preview of the splits

In [ ]:
preview_jsonl(TRAIN_PATH, n=1)
preview_jsonl(VALID_PATH, n=1)
preview_jsonl(TEST_PATH, n=1)

## 5) Create OpenAI client

This cell expects your API key to already be available in your environment, for example through:

- `OPENAI_API_KEY` environment variable
- Colab/Jupyter secret handling
- another secure method

Avoid hardcoding secrets directly into the notebook.

In [ ]:
client = OpenAI()
print("OpenAI client initialized.")

## 6) Upload training and validation files

In [ ]:
with open(TRAIN_PATH, "rb") as f:
    train_file = client.files.create(
        file=f,
        purpose="fine-tune"
    )

with open(VALID_PATH, "rb") as f:
    valid_file = client.files.create(
        file=f,
        purpose="fine-tune"
    )

print("Training file ID :", train_file.id)
print("Validation file ID:", valid_file.id)

## 7) Create the supervised fine-tuning job

In [ ]:
job = client.fine_tuning.jobs.create(
    model=BASE_MODEL,
    training_file=train_file.id,
    validation_file=valid_file.id,
    method={
        "type": "supervised",
        "supervised": {
            "hyperparameters": {
                "n_epochs": "auto",
                "batch_size": "auto",
                "learning_rate_multiplier": "auto"
            }
        }
    },
    metadata={
        "project": PROJECT_NAME,
        "train_examples": str(TRAIN_SIZE),
        "valid_examples": str(VALID_SIZE),
        "test_examples": str(TEST_SIZE)
    }
)

print("Fine-tune job ID:", job.id)
print("Initial status:", job.status)

## 8) Check job status once

In [ ]:
job_status = client.fine_tuning.jobs.retrieve(job.id)
job_status

## 9) Poll until the fine-tuning job finishes

In [ ]:
while True:
    status = client.fine_tuning.jobs.retrieve(job.id)
    print(f"Status: {status.status}")

    if status.status in ["succeeded", "failed", "cancelled"]:
        break

    time.sleep(20)

print("\nFinal status:")
print(status)

if getattr(status, "fine_tuned_model", None):
    print("Fine-tuned model:", status.fine_tuned_model)

## 10) Optional: inspect fine-tuning events

In [ ]:
events = client.fine_tuning.jobs.list_events(fine_tuning_job_id=job.id)
events

## 11) Load the held-out test split

The test set is **not** uploaded to the fine-tuning job. It stays separate for your own evaluation.

In [ ]:
test_rows = load_jsonl(TEST_PATH)
print("Loaded test rows:", len(test_rows))

## 12) Evaluate the final fine-tuned model on the test set

This uses exact string match as a simple baseline.
For structured JSON outputs, you may want to add:
- JSON parsing checks
- field-by-field comparison
- semantic scoring

In [ ]:
def run_test_set(model_name, test_rows, max_cases=None):
    total = 0
    exact = 0
    outputs = []

    rows = test_rows if max_cases is None else test_rows[:max_cases]

    for idx, row in enumerate(rows, start=1):
        prompt_messages, target = split_prompt_and_target(row)

        resp = client.chat.completions.create(
            model=model_name,
            messages=prompt_messages
        )

        pred = (resp.choices[0].message.content or "").strip()
        gold = target.strip()

        is_match = pred == gold
        total += 1
        exact += int(is_match)

        outputs.append({
            "index": idx,
            "prediction": pred,
            "target": gold,
            "exact_match": is_match
        })

        if idx % 20 == 0:
            print(f"Processed {idx}/{len(rows)} test examples...")

    return {
        "total": total,
        "exact_match_count": exact,
        "exact_match_rate": exact / total if total else 0.0,
        "outputs": outputs
    }

In [ ]:
if not getattr(status, "fine_tuned_model", None):
    raise RuntimeError(
        "No fine-tuned model found yet. Make sure the job succeeded first."
    )

TEST_MODEL = status.fine_tuned_model

results = run_test_set(TEST_MODEL, test_rows, max_cases=TEST_SIZE)

print("Test total:", results["total"])
print("Exact matches:", results["exact_match_count"])
print("Exact match rate:", results["exact_match_rate"])

## 13) Save evaluation results

In [ ]:
RESULTS_PATH = OUTPUT_DIR / "test_results.json"

with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("Saved test results to:", RESULTS_PATH)

## Notes

- If the fine-tuning job rejects `gpt-5-chat`, the API error message is the source of truth for your account's current model access.
- If your dataset uses JSON-style assistant outputs, exact match may be too strict. Consider a custom evaluator that parses JSON and compares keys and values.
- Keep your API key outside the notebook source.